In [1]:
import pandas as pd

df = pd.read_csv("https://github.com/ellimilial/clinical-trials-negative-efficacy/raw/refs/heads/main/01-all-clinical-trials.csv.zip")
print(f"Loaded a total of {len(set(df.nct_id))} trials")
print(f"  - {len(df)} trial-intervention-condition")
print(f"  - {len(set(df.condition_name))} conditions")
print(f"  - {len(set(df.intervention_name))} interventions")

df.describe()

Loaded a total of 582141 trials
  - 1944960 trial-intervention-condition
  - 129379 conditions
  - 513773 interventions


,nct_id,brief_title,official_title,overall_status,start_date,completion_date,study_type,phase,why_stopped,condition_name,intervention_name,intervention_type
count,1944960,1944960,1920708,1944960,1933161,1874901,1943989,900713,177518,1943950,1835012,1835284
unique,582141,579644,568025,14,9212,10310,3,7,32998,129378,513772,11
top,NCT03878524,Serial Measurements of Molecular and Architect...,Serial Measurements of Molecular and Architect...,COMPLETED,2014-01-31,2026-12-31,INTERVENTIONAL,PHASE2,Low accrual,Healthy,Placebo,DRUG
freq,2800,2800,2800,990709,6317,32590,1623356,298082,3869,21875,53986,686124


In [2]:
# Only evaluate stopping reasons where we can link to both condition and intervention.
df_filtered = df[pd.notna(df["why_stopped"]) & pd.notna(df["condition_name"]) & pd.notna(df["intervention_name"])]
# Since the model was trained on case-sensitive data we should not be collapsing by case.
df_filtered["why_stopped"].drop_duplicates().reset_index(drop=True)

,why_stopped
0,Replaced by another study.
1,Study never opened; never enrolled participants
2,Original P.I. left the institution
3,Unable to recruit adequate number of subjects
4,Study was never funded.
...,...
31114,etik onay alındığı tarihte yapıldı ve tamamlandı
31115,Participant recruitment has been stopped becau...
31116,This study has been transferred to the Human R...
31117,Recruitment was temporarily suspended pending ...


In [3]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import expit, softmax
from tqdm.auto import tqdm

# -------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------
MODEL_ID = "ellimilial/clinical-trials-negative-efficacy-biomedbert"
TEXT_COL = "why_stopped"
BATCH_SIZE = 32

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# -------------------------------------------------------------------
# Load model
# -------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
model.to(device)
model.eval()

id2label = model.config.id2label
print("Model labels:", id2label)

# -------------------------------------------------------------------
# Prepare unique stopping reasons
# -------------------------------------------------------------------
why_stopped_unique = (
    df_filtered[[TEXT_COL]]
    .drop_duplicates()
    .reset_index(drop=True)
)

texts = why_stopped_unique[TEXT_COL].astype(str).tolist()

# -------------------------------------------------------------------
# Predict
# -------------------------------------------------------------------
all_probs = []
all_preds = []

with torch.no_grad():
    for i in tqdm(range(0, len(texts), BATCH_SIZE)):
        batch_texts = texts[i : i + BATCH_SIZE]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(device)

        outputs = model(**inputs)
        logits = outputs.logits.detach().cpu().numpy()

        # Binary classifier with one output logit
        if logits.shape[1] == 1:
            probs_positive = expit(logits[:, 0])
            preds = (probs_positive >= 0.5).astype(int)

        # Binary/multiclass classifier with 2+ logits
        else:
            probs = softmax(logits, axis=1)

            # Assume positive class is label 1 unless config says otherwise
            probs_positive = probs[:, 1] if logits.shape[1] == 2 else probs.max(axis=1)
            preds = probs.argmax(axis=1)

        all_probs.extend(probs_positive.tolist())
        all_preds.extend(preds.tolist())

why_stopped_predictions = why_stopped_unique.copy()
why_stopped_predictions["y_prob"] = all_probs
why_stopped_predictions["y_pred"] = all_preds
why_stopped_predictions["predicted_label"] = [
    id2label.get(int(pred), str(pred)) for pred in all_preds
]

why_stopped_predictions.head()

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model labels: {0: 'not_negative_efficacy', 1: 'negative_efficacy'}


  0%|          | 0/973 [00:00<?, ?it/s]

,why_stopped,y_prob,y_pred,predicted_label
0,Replaced by another study.,0.000113,0,not_negative_efficacy
1,Study never opened; never enrolled participants,0.000108,0,not_negative_efficacy
2,Original P.I. left the institution,0.000103,0,not_negative_efficacy
3,Unable to recruit adequate number of subjects,0.000103,0,not_negative_efficacy
4,Study was never funded.,0.000105,0,not_negative_efficacy


In [4]:
df_scored = df_filtered.merge(
    why_stopped_predictions,
    on="why_stopped",
    how="left",
)

df_scored[
    [
        "nct_id",
        "condition_name",
        "intervention_name",
        "why_stopped",
        "y_prob",
        "y_pred",
        "predicted_label",
    ]
].head()

,nct_id,condition_name,intervention_name,why_stopped,y_prob,y_pred,predicted_label
0,NCT00000105,Cancer,Biosyn KLH,Replaced by another study.,0.000113,0,not_negative_efficacy
1,NCT00000105,Cancer,Intracel KLH Vaccine,Replaced by another study.,0.000113,0,not_negative_efficacy
2,NCT00000105,Cancer,Montanide ISA51,Replaced by another study.,0.000113,0,not_negative_efficacy
3,NCT00000105,Cancer,Tetanus toxoid,Replaced by another study.,0.000113,0,not_negative_efficacy
4,NCT00000270,Cocaine-Related Disorders,Cocaine,Study never opened; never enrolled participants,0.000108,0,not_negative_efficacy


In [6]:
why_stopped_predictions.to_csv("unique_why_stopped_predictions.csv", index=False)
df_scored.to_csv("02-negative-efficacy-prediction.csv.zip", index=False, compression="zip")

print("Saved:")
print(" - unique_why_stopped_predictions.csv")
print(" - 02-negative-efficacy-prediction.csv.zip")

Saved:
 - unique_why_stopped_predictions.csv
 - 02-negative-efficacy-prediction.csv.zip
